# Player Context Features

This notebook builds historical player-level features for the 100 target matches, using only Ranked Solo/Duo games that occurred before each target match.

# 1. Load Target Matches and Players

In [13]:
import os
import json
import pandas as pd
from collections import Counter

df = pd.read_parquet(
    "data/processed/timeline_features_16_18_master.parquet"
)

target_match_ids = set(df["match_id"].unique())
MATCH_DIR = "data/raw/matches"

player_targets = []

for filename in os.listdir(MATCH_DIR):
    if not filename.endswith(".json"):
        continue

    match_id = filename.replace(".json", "")
    if match_id not in target_match_ids:
        continue

    match_path = os.path.join(MATCH_DIR, filename)
    with open(match_path, "r") as file:
        match_data = json.load(file)

    game_start = match_data["info"]["gameStartTimestamp"]
    for participant in match_data["info"]["participants"]:
        player_targets.append({
            "match_id": match_id,
            "game_start": game_start,
            "puuid": participant["puuid"],
            "team_id": participant["teamId"],
            "champion": participant["championName"],
            "win": int(participant["win"])
        })

player_targets_df = pd.DataFrame(player_targets)

print("Player appearances:", len(player_targets_df))
print("Unique players:", player_targets_df["puuid"].nunique())
print("Matches:", player_targets_df["match_id"].nunique())

player_targets_df.head(10)

Player appearances: 1000
Unique players: 871
Matches: 100


,match_id,game_start,puuid,team_id,champion,win
0,NA1_5645237317,1789858833196,lv5yRlgWr033JQoTcdoqSTKc9VMA3kLS1aXRh56lU8UwDi...,100,Yone,0
1,NA1_5645237317,1789858833196,vibkTE5JzB6JV3cpvMoUco7vD1XVx-42_g32Hui5p7yngg...,100,Sylas,0
2,NA1_5645237317,1789858833196,T6GBUVb3RgD436E52qhLxg9hFJzjNYmBL-pLIqmYEJtIWd...,100,Lux,0
3,NA1_5645237317,1789858833196,W-5_yZxxRRAGdFdXi3vfO4tdsANI17jUYA3ds_KdW7xJ14...,100,Yunara,0
4,NA1_5645237317,1789858833196,8rQlmMWxxIL5ZmgHo3fbawxxBcQ4jrcTUd1EIpEwPWZXO-...,100,Lulu,0
5,NA1_5645237317,1789858833196,O-kKazjPoipk68WwH20qCnJ3WNEYjUsfX_0RjrqUypmmWL...,200,Aatrox,1
6,NA1_5645237317,1789858833196,2-GsGYJnr1_9Yqh0_Am0TEvjn-spPerzWt6mJAvMXsIqlg...,200,Hecarim,1
7,NA1_5645237317,1789858833196,3HoQ92lmd4PV64tL0K7kCPI3WveAn-MSg1r3fQBYZI0-IA...,200,Akshan,1
8,NA1_5645237317,1789858833196,6h12DMGf6oDaTYYtotXKqRX_FqSB1IU95kNykjQBjsc5go...,200,Viktor,1
9,NA1_5645237317,1789858833196,Jmw2qxBFJ7O2m-TFZwrrAkFn70OSBfoE1386RMTNk01h8B...,200,Karma,1


In [14]:
player_targets_df["puuid"].value_counts().describe()

count    871.000000
mean       1.148106
std        0.864295
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max       14.000000
Name: count, dtype: float64

In [15]:
player_targets_df["puuid"].value_counts().head(10)

puuid
8F5OcR5ZyQS09lArSXsivTc_zdG3ZeOCRSxfKDOvkOMzOi_2X4onCUT9jDRPR6jVr0v54-NAzW6nHA    14
WvBlyCeDUDo9sON81n7JVHn8zzkGL5Lsp5vVlwiBNCShxe-KrVqY_NIGTTCgYdvxv0nVu7X-SyvRcw    11
ykfY0R9qsm1UyOQd52Fhv5OXYVQOccHteVA6VI5wsPCLSnjv8IUwpeMA3Bt0P0B2lL6clRipc1q_ig     9
5ixDAiJe9YkqaWVUU_Ronc_ZZbstVhZH62gLYBnH_buFW-tmZPsiJqeJXTjXTK8ezHwYqQJcdXDW9g     9
vhKAc-bIYswVWD8wvnUu29q7CuM3VIFopWVYQ7aATPz_Vhzd3JTm1qmZeWGKG44B-XemSUvXDH2oDA     8
fPB-QICIxZ0xaDOt5Q_8iODj7aOTncWs-vTVtngAkIiD2_ZOfHl_KryzPvIeMP3whVei805eaqPyWg     8
Jmw2qxBFJ7O2m-TFZwrrAkFn70OSBfoE1386RMTNk01h8Bn_n4E5UYPb3GZhEIVWW_Q11vPWmGCLvg     8
rpTs4NIGMyjG2nmFoLDEXCSzgcOnBc5F4ibmIMUOUvbfMfTCSS_VDskiYbnKIvsezxWzlTvtCvGbLg     6
QEmknnt8kDkLSsKOLrRweFDfE2_NIv9Udb3zJyz-6viSqfh2doXrxyBWNcjk0xUZRcgMskHEVEATxQ     6
gPYQua__rWBu7xi54rIBRdsagS0NLDvsQFlPqnOzRYemtdObkSYZX7bBrPZuNH4vOXIsGP78KMdOCw     4
Name: count, dtype: int64

# 2. Riot API Setup

In [16]:
import requests
import time
from dotenv import load_dotenv

load_dotenv(override=True)

RIOT_API_KEY = os.getenv("RIOT_API_KEY")

headers = {
    "X-Riot-Token": RIOT_API_KEY
}

ROUTING = "americas"
TARGET_QUEUE = 420
HISTORY_GAMES = 50


def riot_get(url, params=None):
    while True:
        response = requests.get(
            url,
            headers=headers,
            params=params
        )

        if response.status_code == 429:
            wait_time = int(response.headers.get("Retry-After", 10))
            print(f"Rate limited. Waiting {wait_time} seconds...")
            time.sleep(wait_time + 1)
            continue

        response.raise_for_status()
        return response.json()


def get_match(match_id):
    url = (
        f"https://{ROUTING}.api.riotgames.com/"
        f"lol/match/v5/matches/{match_id}"
    )

    return riot_get(url)


match_cache = {}


def get_match_cached(match_id):
    if match_id not in match_cache:
        match_cache[match_id] = get_match(match_id)

    return match_cache[match_id]

# 3. Historical Match Lookup

In [ ]:
def get_prior_match_ids(puuid, before_timestamp_ms, count=HISTORY_GAMES):
    url = (
        f"https://{ROUTING}.api.riotgames.com/"
        f"lol/match/v5/matches/by-puuid/{puuid}/ids"
    )

    params = {
        "queue": TARGET_QUEUE,
        "endTime": (before_timestamp_ms // 1000) - 1
        "start": 0,
        "count": count
    }

    return riot_get(url, params=params)

# 4. Player History Feature Extraction

In [18]:
def get_player_history_features(
    puuid,
    target_champion,
    prior_match_ids
):
    if not prior_match_ids:
        return {
            "prior_games": 0,
            "prior_win_rate": None,
            "main_champion": None,
            "main_champion_share": None,
            "one_trick": 0,
            "playing_one_trick": 0
        }

    wins = 0
    champions = []

    for match_id in prior_match_ids:
        match_data = get_match_cached(match_id)

        participant = next(
            player
            for player in match_data["info"]["participants"]
            if player["puuid"] == puuid
        )

        wins += int(participant["win"])
        champions.append(participant["championName"])

    games_played = len(prior_match_ids)

    if not champions:
        return {
            "prior_games": 0,
            "prior_win_rate": None,
            "main_champion": None,
            "main_champion_share": None,
            "one_trick": 0,
            "playing_one_trick": 0
        }

    champion_counts = Counter(champions)
    main_champion, main_games = champion_counts.most_common(1)[0]

    win_rate = wins / games_played
    main_champion_share = main_games / games_played
    one_trick = int(main_champion_share >= 0.50)
    playing_one_trick = int(
        one_trick == 1
        and target_champion == main_champion
    )

    return {
        "prior_games": games_played,
        "prior_win_rate": win_rate,
        "main_champion": main_champion,
        "main_champion_share": main_champion_share,
        "one_trick": one_trick,
        "playing_one_trick": playing_one_trick
    }

# 5. Test on One Player

In [19]:
test_row = player_targets_df.iloc[0]

test_match_ids = get_prior_match_ids(
    test_row["puuid"],
    test_row["game_start"]
)

print("Historical matches found:", len(test_match_ids))
print(test_match_ids[:5])

test_features = get_player_history_features(
    puuid=test_row["puuid"],
    target_champion=test_row["champion"],
    prior_match_ids=test_match_ids
)

test_features

Historical matches found: 50
['NA1_5645217059', 'NA1_5645131766', 'NA1_5645106397', 'NA1_5644698596', 'NA1_5644672114']


{'prior_games': 50,
 'prior_win_rate': 0.38,
 'main_champion': 'Alistar',
 'main_champion_share': 0.12,
 'one_trick': 0,
 'playing_one_trick': 0}

In [ ]:
print("Cached historical matches:", len(match_cache))

# 6. Save Player Targets

In [20]:
os.makedirs("data/processed", exist_ok=True)

player_targets_df.to_parquet(
    "data/processed/player_targets_16_18_master.parquet",
    index=False
)